# ALPHA构建
对于上述每一个组合，控制市值等因子，计算其ALPHA的显著性        

## 导入库

In [5]:
import warnings
from pathlib import Path
import polars as pl
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import plotly.express as px

## 超参数

In [6]:
import os
import dotenv
dotenv.load_dotenv()
CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "adbc"

TASK_ID_PREFIX = 'baseline1'  # 任务id前缀
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE_BASELINE_REG_DIR = SAVE_BASE_DIR + '/baseline_reg'
SAVE = True # 是否保存数据

RISK_FREE_RATE = 0.015 / 12 # 无风险利率  

## 读取数据(测试)
### 读取FF5数据  

FF5数据是一个月度时序数据，统计了每个月，FF5因子按照2*3方式构建组合的对冲组合的收益。使用该数据来计算alpha  

从json导入（测试）  

In [7]:
# 从 DB 读取 FF5（statics.stk_mkt_fivefacmonth）
ff5 = pl.read_database_uri(
    uri=CONNECTION_URL,
    query="""
        SELECT markettype_id AS "MarkettypeID", trading_month AS "TradingMonth",
               portfolios AS "Portfolios", risk_premium1 AS "RiskPremium1",
               smb1 AS "SMB1", hml1 AS "HML1", rmw1 AS "RMW1", cma1 AS "CMA1"
        FROM statics.stk_mkt_fivefacmonth
    """,
    engine=ENGINE,
)
ff5.head()


MarkettypeID,TradingMonth,Portfolios,RiskPremium1,SMB1,HML1,RMW1,CMA1
str,date,i16,f64,f64,f64,f64,f64
"""P9712""",1997-06-01,2,-0.004782,0.016247,-0.016211,0.020201,-0.05377
"""P9709""",1997-07-01,3,-0.066855,0.047058,0.020976,-0.006972,-0.001247
"""P9709""",1997-10-01,1,0.129622,-0.042278,-0.016642,0.043437,-0.028362
"""P9714""",1997-03-01,3,0.184236,0.063811,-0.011957,0.038304,-0.048847
"""P9701""",1997-06-01,1,-0.037687,0.009417,-0.026915,0.012488,-0.090306


### 读取分桶数据  
- 分桶数据用来和FF5数据合并，计算alpha     
- 分桶表现数据用于合并alpha表现  



In [8]:
combined_series = pl.scan_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶市值加权收益.parquet')
performance = pl.read_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶表现.parquet')

## 处理数据：  

- 1.选用Portfolio == 1 的组合 (2*3构建)  
- 2.MarkettypeID == P9725：沪深A股和创业板和科创板 (不选京，否则没有早期年份)     
- 3.去除上述两列  
- 4.将TradingMonth列重命名为`date`    
- 5.因子重命名为`market_ret, smb, hml, rmw, cma`  

In [9]:
ff5 = ff5.filter((pl.col('Portfolios') == 1) & (pl.col('MarkettypeID') == 'P9714')).select(['TradingMonth', 'RiskPremium1', 'SMB1', 'HML1', 'RMW1', 'CMA1'])
ff5 = ff5.rename({'TradingMonth':'date', 'RiskPremium1':'market_ret', 'SMB1':'smb', 'HML1':'hml', 'RMW1':'rmw', 'CMA1':'cma'})
ff5 = ff5.lazy()

In [10]:
ff5.head().collect()

date,market_ret,smb,hml,rmw,cma
date,f64,f64,f64,f64,f64
1997-04-01,0.08511,-0.047831,-0.028225,0.000637,-0.166661
1997-08-01,-0.001449,0.008266,0.047883,-0.082115,0.077224
1997-09-01,-0.116536,0.022238,-0.007937,-0.00569,-0.001345
1997-07-01,-0.066855,0.045986,0.012023,0.004122,0.014512
1997-06-01,-0.004782,0.015851,0.008128,0.044233,-0.054761


## 计算alpha  

### CAPM-ALPHA    
从FF5中获取market_ret列，形成时序数据`date-market_ret`  
将数据和combined_series合并，形成`date-code-bucket_id-ret-market_ret`表  

计算`(ret-risk_free_rate)~market_ret`回归的alpha（NW标准误）   

In [11]:
# 获取date-market_ret
market_ret = ff5.select('date','market_ret')

# 合并
joined_series = combined_series.join(market_ret, on='date', how='left')

## 按 bucket_id 分组回归（Polars map_groups）
coll = joined_series.collect()

In [12]:
def regress_one(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]  # 安全取分组 id，避免 Polars 内索引触发 panic
    bid = str(bid) 
    try:
        y = g['weighted_sum_ret'].to_numpy() - RISK_FREE_RATE
        x = g['market_ret'].to_numpy()
        x_with_constant = sm.add_constant(x)
        model = sm.OLS(y, x_with_constant)
        results = model.fit() if len(y) < 10 else model.fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        return pl.DataFrame({
            'bucket_id': [bid],
            'camp-alpha': [float(results.params[0])],
            't': [float(results.tvalues[0])],
            'p': [float(results.pvalues[0])],
        })
    except Exception as e:
        import traceback
        traceback.print_exc()
        warnings.warn(f"计算{bid}时发生错误: {e}")
        return pl.DataFrame({
            'bucket_id': [bid],
            'camp-alpha': [0.0],
            't': [0.0],
            'p': [1.0],
        })

alpha_table = coll.group_by('bucket_id').map_groups(regress_one)

格式化输出：   
[ ]内为HAC-t，()内为p值    

In [13]:
# 格式化输出（4 位有效数字）
alpha_table = alpha_table.with_columns(
    pl.col('camp-alpha').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('camp-alpha'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)

# 为t和p添加括号  
alpha_table = alpha_table.select(
    pl.col('bucket_id'),
    pl.col('camp-alpha'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)

# 将alpha、t、p居中对齐
alpha_table = alpha_table.with_columns(
    pl.col('camp-alpha').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('camp-alpha'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)

# 将alpha、t、p合并为一行
alpha_table = alpha_table.select(
    pl.col('bucket_id'),
    (pl.col('camp-alpha') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('camp-alpha'),
)

alpha_table = alpha_table.sort('bucket_id')

with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(alpha_table)

bucket_id,camp-alpha
str,str
"""0""",""" -0.003451 [-4.671] (2.996e-06) """
"""1""",""" -0.002243 [-5.388] (7.141e-08) """
"""2""",""" -0.001109 [-3.294] (0.0009869) """
"""3""",""" -0.0008111 [-2.622] (0.008738) """
"""4""",""" -0.0007193 [-2.49] (0.01276) """
…,…
"""6""",""" -0.000583 [-2.223] (0.02621) """
"""7""",""" -6.649e-05 [-0.2005] (0.8411) """
"""8""",""" -0.0002166 [-0.7037] (0.4816) """


### FF3-ALPHA  
使用FAMA-FRENCH模型计算ALPHA  

与CAPM类似，使用`market_ret, smb, hml, rmw, cma`作为自变量，计算`weighted_sum_ret`的alpha    

In [14]:
# 获取ff3数据
ff3 = ff5.select('date', 'market_ret', 'smb', 'hml')

#连接
joined_series = combined_series.join(ff3, on='date', how='left')

# 按bucket_id分组回归
coll = joined_series.collect()

def regress_ff3(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]
    try:
        y = g['weighted_sum_ret'].to_numpy() - RISK_FREE_RATE
        x = g[['market_ret', 'smb', 'hml']].to_numpy()
        x_with_constant = sm.add_constant(x)
        results = sm.OLS(y, x_with_constant).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff3-alpha': [float(results.params[0])],
            't': [float(results.tvalues[0])],
            'p': [float(results.pvalues[0])],
        })
    except Exception as e:
        warnings.warn(f"计算{bid}时发生错误: {e}")
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff3-alpha': [0.0],
            't': [0.0],
            'p': [1.0],
        })

ff3_alpha_table = coll.group_by('bucket_id').map_groups(regress_ff3)

格式化输出

In [15]:
# FF3-alpha 格式化输出（4 位有效数字、括号、居中对齐、合并一行）
ff3_alpha_table = ff3_alpha_table.with_columns(
    pl.col('ff3-alpha').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('ff3-alpha'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)
ff3_alpha_table = ff3_alpha_table.select(
    pl.col('bucket_id'),
    pl.col('ff3-alpha'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)
ff3_alpha_table = ff3_alpha_table.with_columns(
    pl.col('ff3-alpha').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('ff3-alpha'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)
ff3_alpha_table = ff3_alpha_table.select(
    pl.col('bucket_id'),
    (pl.col('ff3-alpha') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('ff3-alpha'),
)

ff3_alpha_table = ff3_alpha_table.sort('bucket_id')

with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(ff3_alpha_table)


bucket_id,ff3-alpha
str,str
"""0""",""" -0.003441 [-4.743] (2.106e-06) """
"""1""",""" -0.00211 [-5.275] (1.329e-07) """
"""2""",""" -0.00104 [-3.204] (0.001356) """
"""3""",""" -0.0007225 [-2.354] (0.01859) """
"""4""",""" -0.0006516 [-2.333] (0.01963) """
…,…
"""6""",""" -0.0005267 [-2.121] (0.0339) """
"""7""",""" -2.787e-05 [-0.08817] (0.9297) """
"""8""",""" -0.0001413 [-0.4704] (0.6381) """


### FF5-ALPHA 

In [16]:
# 获取 FF5 数据（五因子）
ff5_factors = ff5.select('date', 'market_ret', 'smb', 'hml', 'rmw', 'cma')

# 连接
joined_series = combined_series.join(ff5_factors, on='date', how='left')

# 按 bucket_id 分组回归
coll = joined_series.collect()

def regress_ff5(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]
    try:
        y = g['weighted_sum_ret'].to_numpy() - RISK_FREE_RATE
        x = g[['market_ret', 'smb', 'hml', 'rmw', 'cma']].to_numpy()
        x_with_constant = sm.add_constant(x)
        model = sm.OLS(y, x_with_constant)
        results = model.fit() if len(y) < 10 else model.fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff5-alpha': [float(results.params[0])],
            't': [float(results.tvalues[0])],
            'p': [float(results.pvalues[0])],
        })
    except Exception as e:
        warnings.warn(f"计算{bid}时发生错误: {e}")
        return pl.DataFrame({
            'bucket_id': [bid],
            'ff5-alpha': [0.0],
            't': [0.0],
            'p': [1.0],
        })

ff5_alpha_table = coll.group_by('bucket_id').map_groups(regress_ff5)

格式化输出

In [17]:
# FF5-alpha 格式化输出（4 位有效数字、括号、居中对齐、合并一行）
ff5_alpha_table = ff5_alpha_table.with_columns(
    pl.col('ff5-alpha').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('ff5-alpha'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)
ff5_alpha_table = ff5_alpha_table.select(
    pl.col('bucket_id'),
    pl.col('ff5-alpha'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)
ff5_alpha_table = ff5_alpha_table.with_columns(
    pl.col('ff5-alpha').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('ff5-alpha'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)
ff5_alpha_table = ff5_alpha_table.select(
    pl.col('bucket_id'),
    (pl.col('ff5-alpha') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('ff5-alpha'),
)

ff5_alpha_table = ff5_alpha_table.sort('bucket_id')

with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(ff5_alpha_table)

bucket_id,ff5-alpha
str,str
"""0""",""" -0.003285 [-4.565] (4.997e-06) """
"""1""",""" -0.002005 [-5.044] (4.567e-07) """
"""2""",""" -0.0009266 [-2.901] (0.003725) """
"""3""",""" -0.0006258 [-2.043] (0.04103) """
"""4""",""" -0.0005569 [-2.013] (0.0441) """
…,…
"""6""",""" -0.000447 [-1.805] (0.07111) """
"""7""",""" 0.0001017 [0.3142] (0.7533) """
"""8""",""" -5.775e-05 [-0.1939] (0.8462) """


## 合并所有表现

In [18]:
performance_all = performance

In [ ]:
performance_all = performance_all.join(alpha_table, on='bucket_id', how='left')
performance_all = performance_all.join(ff3_alpha_table, on='bucket_id', how='left')
performance_all = performance_all.join(ff5_alpha_table, on='bucket_id', how='left')
performance_all = performance_all.sort('bucket_id')

bucket_id,mean_return,sharp,camp-alpha,ff3-alpha,ff5-alpha
str,str,str,str,str,str
"""0""",""" -0.0022 [-3.011] (0.…","""-0.3798""",""" -0.003451 [-4.671] (2.9…",""" -0.003441 [-4.743] (2.1…",""" -0.003285 [-4.565] (4.9…"
"""1""",""" -0.000957 [-2.158] (0.…","""-0.3661""",""" -0.002243 [-5.388] (7.1…",""" -0.00211 [-5.275] (1.3…",""" -0.002005 [-5.044] (4.5…"
"""2""",""" 0.0001727 [0.4647] (0…","""-0.2068""",""" -0.001109 [-3.294] (0.0…",""" -0.00104 [-3.204] (0.…",""" -0.0009266 [-2.901] (0.…"
"""3""",""" 0.0005049 [1.41] (0…","""-0.1562""",""" -0.0008111 [-2.622] (0.…",""" -0.0007225 [-2.354] (0.…",""" -0.0006258 [-2.043] (0.…"
"""4""",""" 0.0005588 [1.814] (0.…","""-0.1558""",""" -0.0007193 [-2.49] (0.…",""" -0.0006516 [-2.333] (0.…",""" -0.0005569 [-2.013] (0…"


In [31]:
with pl.Config(
    tbl_rows=50,           # 显示所有行（不截断高度）
    tbl_cols=50,           # 显示所有列
    tbl_width_chars=200,     # 不限制表格总宽度
    fmt_str_lengths=200,     # 不限制字符串长度
):
    display(performance_all)  # 显示整个完整的表


bucket_id,mean_return,sharp,camp-alpha,ff3-alpha,ff5-alpha
str,str,str,str,str,str
"""0""",""" -0.0022 [-3.011] (0.002605) ""","""-0.3798""",""" -0.003451 [-4.671] (2.996e-06) """,""" -0.003441 [-4.743] (2.106e-06) """,""" -0.003285 [-4.565] (4.997e-06) """
"""1""",""" -0.000957 [-2.158] (0.03092) ""","""-0.3661""",""" -0.002243 [-5.388] (7.141e-08) """,""" -0.00211 [-5.275] (1.329e-07) """,""" -0.002005 [-5.044] (4.567e-07) """
"""2""",""" 0.0001727 [0.4647] (0.6421) ""","""-0.2068""",""" -0.001109 [-3.294] (0.0009869) """,""" -0.00104 [-3.204] (0.001356) """,""" -0.0009266 [-2.901] (0.003725) """
"""3""",""" 0.0005049 [1.41] (0.1586) ""","""-0.1562""",""" -0.0008111 [-2.622] (0.008738) """,""" -0.0007225 [-2.354] (0.01859) """,""" -0.0006258 [-2.043] (0.04103) """
"""4""",""" 0.0005588 [1.814] (0.06962) ""","""-0.1558""",""" -0.0007193 [-2.49] (0.01276) """,""" -0.0006516 [-2.333] (0.01963) """,""" -0.0005569 [-2.013] (0.0441) """
"""5""",""" 0.0004719 [1.676] (0.09371) ""","""-0.1895""",""" -0.0008 [-3.057] (0.002235) """,""" -0.0007195 [-2.952] (0.003159) """,""" -0.0006547 [-2.67] (0.007591) """
"""6""",""" 0.000739 [2.425] (0.01529) ""","""-0.1295""",""" -0.000583 [-2.223] (0.02621) """,""" -0.0005267 [-2.121] (0.0339) """,""" -0.000447 [-1.805] (0.07111) """
"""7""",""" 0.001259 [3.207] (0.00134) ""","""0.001923""",""" -6.649e-05 [-0.2005] (0.8411) """,""" -2.787e-05 [-0.08817] (0.9297) """,""" 0.0001017 [0.3142] (0.7533) """
"""8""",""" 0.001104 [3.049] (0.002293) ""","""-0.03321""",""" -0.0002166 [-0.7037] (0.4816) """,""" -0.0001413 [-0.4704] (0.6381) """,""" -5.775e-05 [-0.1939] (0.8462) """


In [20]:
if SAVE:
    performance_all.write_parquet(SAVE_BASELINE_REG_DIR + '/基准回归-分桶表现+alpha.parquet')